In [3]:
from imap_processing.quality_flags import ImapDEScatteringUltraFlags
from imap_processing.ultra.l1b.lookup_utils import get_scattering_coefficients
from imap_processing.ultra.l1c.ultra_l1c_pset_bins import build_energy_bins
from imap_processing.cdf.imap_cdf_manager import ImapCdfAttributes
from imap_processing.cdf.utils import write_cdf
import numpy as np
import pandas as pd
import os
import xarray as xr
import spiceypy as sp



# TODO CHANGE SENSOR FOR 45 or 90
sensor=45
# ruff: noqa: E501
cdf_manager = ImapCdfAttributes()
cdf_manager.add_instrument_global_attrs("ultra")
cdf_manager.add_instrument_variable_attrs("ultra", "l1b")

folder_path = (
    "/Users/luco3133/projects/ultra_stuff/validation_stuff"
    f"/other_var_validation_20251024/ultra-{sensor}-inputs"
)

kernles = [
    "/Users/luco3133/projects/imap_processing/imap_processing/"
    "tests/spice/test_data/imap_sclk_0000.tsc",
    "/Users/luco3133/projects/imap_processing/imap_processing/"
    "tests/spice/test_data/naif0012.tls",
]


def vectorijk_to_theta_phi(x_inst, y_inst, z_inst):
    """Get theta and phi."""
    x_permuted = z_inst  # K
    y_permuted = y_inst  # J
    z_permuted = x_inst  # I

    # Declination (theta)
    theta_rad = np.arcsin(np.clip(z_permuted, -1.0, 1.0))

    # Right Ascension (phi)
    phi_rad = np.arctan2(y_permuted, x_permuted)

    # Convert to degrees
    theta_deg = np.degrees(theta_rad)
    phi_deg = np.degrees(phi_rad)

    return theta_deg, phi_deg


def get_scattering_threshold_for_energy(energy):
    """Get scattering threshold based on energy."""
    if energy < 5:
        return np.inf
    elif energy < 8:
        return np.inf
    elif energy < 10:
        return np.inf
    elif energy < 20:
        return np.inf
    elif energy < 400:
        return np.inf


files = os.listdir(folder_path)
pointings = sorted(np.unique([f.split("-")[-1].replace(".csv", "") for f in files]))
#pointings = ["p0"]

# You'll need to provide ancillary_files - adjust path as needed
ancillary_files = {
    "l1b-90sensor-scattering-calibration-data": "/Users/luco3133/projects/imap_processing/data/imap/ancillary/ultra/imap_ultra_l1b-90sensor-scattering-calibration-data_20250101_v000.csv",
    "l1b-scattering-thresholds-per-energy": "/Users/luco3133/projects/imap_processing/data/imap/ancillary/ultra/imap_ultra_l1b-scattering-thresholds-per-energy_20250101_v000.csv",
}
print(pointings)
with sp.KernelPool(kernles) as pool:
    for pointing in pointings:
        for id in [sensor]:
            print(
                f"Creating direct event dataset for {pointing} pointing and {id} sensor"
            )

            ae_flux = pd.read_csv(
                os.path.join(folder_path, f"AE-IMAP_ULTRA_{id}-{pointing}.csv")
            )
            rates_flux = pd.read_csv(
                f"{folder_path}/Rates-IMAP_ULTRA_{id}-{pointing}.csv"
            )

            # energy_max = 1000

            # Get event energies
            #TODO CHANGE TO ENERGY_SC IF SPACECRAFT
            event_energies = ae_flux["energy_hel"].values

            print(f"  Processing {len(event_energies)} events")

            # Calculate theta/phi from instrument direction vectors using Java's method
            print(
                "  Calculating theta/phi from instrument direction vectors (Java method)..."
            )
            x_inst = ae_flux["x_inst"].values
            y_inst = ae_flux["y_inst"].values
            z_inst = ae_flux["z_inst"].values

            theta, phi = vectorijk_to_theta_phi(x_inst, y_inst, z_inst)

            print(f"  Theta range: [{np.min(theta):.2f}, {np.max(theta):.2f}] deg")
            print(f"  Phi range: [{np.min(phi):.2f}, {np.max(phi):.2f}] deg")

            # Apply scattering filter using energy bin geometric means
            print("\n  === APPLYING SCATTERING FILTER ===")

            # Get energy bins
            intervals, _, energy_bin_geometric_means = build_energy_bins()
            print(intervals)
            intervals = np.array(intervals)
            energy_bin_geometric_means = np.array(energy_bin_geometric_means)

            # FILTER: Only process events within the valid energy range
            energy_in_range = (event_energies >= intervals[0, 0]) & (
                event_energies <= intervals[-1, 1]
            )
            print(f"Energy range: {intervals[0, 0]:.1f} - {intervals[-1, 1]:.1f} keV")
            print(f"Events in range: {np.sum(energy_in_range)}/{len(event_energies)}")

            # Initialize quality flags for ALL events (keep original size)
            # Flag events outside energy range as outliers
            # Initialize quality flags for ALL events (keep original size)
            scattering_quality_flags = np.zeros(
                len(event_energies), dtype=np.uint16
            )  # ← ADD THIS

            # Flag events outside energy range as outliers
            outlier_quality_flags = np.zeros(len(event_energies), dtype=np.uint16)
            outlier_quality_flags[~energy_in_range] = 1
            # np.testing.assert_array_equal(outlier_quality_flags[:9], ~[False, False, True, False, False, True, False, True, True])
            print(f"Outlier quality flags: {outlier_quality_flags}")
            # Determine which energy bin each event falls into (for all events)
            bin_indices = np.digitize(event_energies, intervals[:, 0]) - 1
            bin_indices = np.clip(bin_indices, 0, len(energy_bin_geometric_means) - 1)

            print(f"Bin indices: {bin_indices}")
            # Process each energy bin
            for ebin in range(len(energy_bin_geometric_means)):
                # Get events in THIS bin only
                event_mask = (bin_indices == ebin) & energy_in_range

                if not np.any(event_mask):
                    continue

                # Get energy bin geometric mean
                energy_geom_mean = energy_bin_geometric_means[ebin]

                # Get threshold for this energy
                threshold = get_scattering_threshold_for_energy(energy_geom_mean)

                # Get scattering coefficients for events in this bin
                theta_coeffs, phi_coeffs = get_scattering_coefficients(
                    theta[event_mask],
                    phi[event_mask],
                    lookup_tables=None,
                    ancillary_files=ancillary_files,
                    instrument_id=id,
                )
                # After getting coefficients
                has_nan_coeff = (
                    np.isnan(theta_coeffs[:, 0])
                    | np.isnan(theta_coeffs[:, 1])
                    | np.isnan(phi_coeffs[:, 0])
                    | np.isnan(phi_coeffs[:, 1])
                )

                # Flag events with NaN coefficients
                event_indices = np.where(event_mask)[0]
                scattering_quality_flags[event_indices[has_nan_coeff]] |= (
                    ImapDEScatteringUltraFlags.NAN_PHI_OR_THETA.value
                )

                # For valid coefficients, calculate FWHM and check threshold
                valid_coeffs = ~has_nan_coeff
                if np.any(valid_coeffs):
                    fwhm_theta_valid = (
                        theta_coeffs[valid_coeffs, 0]
                        * energy_geom_mean ** theta_coeffs[valid_coeffs, 1]
                    )
                    fwhm_phi_valid = (
                        phi_coeffs[valid_coeffs, 0]
                        * energy_geom_mean ** phi_coeffs[valid_coeffs, 1]
                    )

                    theta_exceeds = fwhm_theta_valid > threshold
                    phi_exceeds = fwhm_phi_valid > threshold
                    either_exceeds = theta_exceeds | phi_exceeds

                    # Get indices of valid events that exceed
                    valid_event_indices = event_indices[valid_coeffs]
                    scattering_quality_flags[valid_event_indices[either_exceeds]] |= (
                        ImapDEScatteringUltraFlags.ABOVE_THRESHOLD.value
                    )
                # Calculate FWHM using energy bin geometric mean
                fwhm_theta = theta_coeffs[:, 0] * energy_geom_mean ** theta_coeffs[:, 1]
                fwhm_phi = phi_coeffs[:, 0] * energy_geom_mean ** phi_coeffs[:, 1]
                # Check for NaN values

                # Don't flag NaN FWHM - treat NaN as if it passes
                has_nan_fwhm = np.isnan(fwhm_theta) | np.isnan(fwhm_phi)

                # Only check threshold for non-NaN events
                theta_exceeds = np.zeros(len(fwhm_theta), dtype=bool)
                phi_exceeds = np.zeros(len(fwhm_phi), dtype=bool)

                valid = ~has_nan_fwhm
                theta_exceeds[valid] = fwhm_theta[valid] > threshold
                phi_exceeds[valid] = fwhm_phi[valid] > threshold

                either_exceeds = theta_exceeds | phi_exceeds
                # has_nan = np.isnan(fwhm_theta) | np.isnan(fwhm_phi)
                # either_exceeds = either_exceeds | has_nan

                event_indices = np.where(event_mask)[0]
                scattering_quality_flags[event_indices[either_exceeds]] |= (
                    ImapDEScatteringUltraFlags.ABOVE_THRESHOLD.value
                )

                # Debug output
                n_flagged = np.sum(either_exceeds)

                # Debug output for bin 0
                if ebin == 0:
                    n_events = np.sum(event_mask)
                    n_flagged = np.sum(either_exceeds)
                    has_nan = np.isnan(fwhm_theta) | np.isnan(fwhm_phi)

                    # Check accidentals
                    bin_0_accidentals = ae_flux["accidental"].values[event_mask]
                    n_accidentals = np.sum(bin_0_accidentals)

                    if ebin == 0:
                        has_nan = np.isnan(fwhm_theta) | np.isnan(fwhm_phi)

                        # For NaN events, check their theta/phi values
                        if np.any(has_nan):
                            nan_indices = np.where(has_nan)[0]
                            print("\n  NaN FWHM Debug:")
                            print(f"  Total NaN events: {np.sum(has_nan)}")

                            # Get original indices in full array
                            original_indices = np.where(event_mask)[0][nan_indices]
                            theta_nan = theta[original_indices]
                            phi_nan = phi[original_indices]

                            print(
                                f"  Theta for NaN events: min={theta_nan.min():.4f}, max={theta_nan.max():.4f}"
                            )
                            print(
                                f"  Phi for NaN events: min={phi_nan.min():.4f}, max={phi_nan.max():.4f}"
                            )

                            # Check coefficients
                            print("  First 5 NaN coefficients:")
                            for i in range(min(5, len(nan_indices))):
                                idx = nan_indices[i]
                                print(
                                    f"    theta={theta[original_indices[i]]:.2f}, phi={phi[original_indices[i]]:.2f}, "
                                    f"a_theta={theta_coeffs[idx, 0]:.4f}, g_theta={theta_coeffs[idx, 1]:.4f}"
                                )
                                # Check bin 0 specifically
                                bin_0_mask = (bin_indices == 0) & energy_in_range
                                bin_0_flagged = bin_0_mask & (
                                    (scattering_quality_flags > 0)
                                    | (outlier_quality_flags > 0)
                                )
                                print(
                                    f"  Energy bin 0 (in range): {np.sum(bin_0_mask)} events, {np.sum(bin_0_flagged)} flagged, {np.sum(bin_0_mask) - np.sum(bin_0_flagged)} passing"
                                )
                                print("  ===================================\n")

            # Get scatter values (for reference only)
            scatter_values = ae_flux["scatter"].values.copy()

            # Set ebin
            # ebin = np.where(
            #     (ae_flux["accidental"].values & (ae_flux["energy_sc"] > energy_max)),
            #     255,
            #     1,
            # )
            ebin = np.ones_like(ae_flux["accidental"].values.astype(np.uint16))

            # Now create dataset with ALL original data (no filtering)
            l1b_ds = xr.Dataset(
                {
                    "event_times": (
                        ["epoch"],
                        ae_flux["tdb (s)"].values.astype(np.float32),
                    ),
                    "velocity_sc": (
                        ["epoch", "component"],
                        np.stack(
                            [
                                ae_flux["v_x_sc"].values,
                                ae_flux["v_y_sc"].values,
                                ae_flux["v_z_sc"].values,
                            ],
                            axis=1,
                        ).astype(np.float32),
                    ),
                    "velocity_dps_sc": (
                        ["epoch", "component"],
                        np.stack(
                            [
                                ae_flux["v_x_sc"].values,
                                ae_flux["v_y_sc"].values,
                                ae_flux["v_z_sc"].values,
                            ],
                            axis=1,
                        ).astype(np.float32),
                    ),
                    "velocity_dps_helio": (
                        ["epoch", "component"],
                        np.stack(
                            [
                                ae_flux["v_x_hel"].values,
                                ae_flux["v_y_hel"].values,
                                ae_flux["v_z_hel"].values,
                            ],
                            axis=1,
                        ).astype(np.float32),
                    ),
                    "energy_spacecraft": (
                        ["epoch"],
                        ae_flux["energy_sc"].values.astype(np.float32),
                    ),
                    "energy_heliosphere": (
                        ["epoch"],
                        ae_flux["energy_hel"].values.astype(np.float32),
                    ),
                    "energy": (
                        ["epoch"],
                        ae_flux["energy_sc"].values.astype(np.float32),
                    ),
                    "tof_energy": (
                        ["epoch"],
                        ae_flux["energy_sc"].values.astype(np.float32),
                    ),
                    "event_efficiency": (
                        ["epoch"],
                        ae_flux["eff"].values.astype(np.float64),
                    ),
                    "geometric_factor_blades": (
                        ["epoch"],
                        ae_flux["gf"].values.astype(np.float64),
                    ),
                    "quality_scattering": (
                        ["epoch"],
                        scattering_quality_flags,
                    ),
                    "quality_outliers": (
                        ["epoch"],
                        outlier_quality_flags,
                    ),
                    "scatter": (
                        ["epoch"],
                        scatter_values.astype(np.float32),
                    ),
                    "ebin": (
                        ["epoch"],
                        ebin,
                    ),
                    "theta": (
                        ["epoch"],
                        theta.astype(np.float32),
                    ),
                    "phi": (
                        ["epoch"],
                        phi.astype(np.float32),
                    ),
                },
                coords={
                    "epoch": ae_flux["tdb (s)"].values.astype(np.float64),
                    "component": ["x", "y", "z"],
                },
                attrs=cdf_manager.get_global_attributes(
                    f"imap_ultra_l1b_{id}sensor-de"
                ),
            )

            # rates_ds = xr.Dataset(
            #     {
            #         "spin_phase": (
            #             ["epoch"],
            #             rates_flux["Spin Phase (deg)"].values.astype(np.float32),
            #         ),
            #         "start_rate": (
            #             ["epoch"],
            #             rates_flux["Start Rate (Hz)"].values.astype(np.float32),
            #         ),
            #         "stop_rate": (
            #             ["epoch"],
            #             rates_flux["Stop Rate (Hz)"].values.astype(np.float32),
            #         ),
            #         "coin_rate": (
            #             ["epoch"],
            #             rates_flux["Coin Rate (Hz)"].values.astype(np.float32),
            #         ),
            #         "dead_time_ratio": (
            #             ["epoch"],
            #             rates_flux["Dead Time Ratio"].values.astype(np.float32),
            #         ),
            #     },
            #     coords={
            #         "epoch": np.arange(len(rates_flux["Dead Time Ratio"].values)),
            #     },
            #     attrs=cdf_manager.get_global_attributes(
            #         f"imap_ultra_l1a_{id}sensor-rates"
            #     ),
            # )

            print("Datasets created")
            l1b_ds.attrs["Data_version"] = "001"
            # rates_ds.attrs["Data_version"] = "000"
            l1b_ds.attrs["Repointing"] = f"repoint{int(pointing.replace('p', '')):05d}"
            # rates_ds.attrs["Repointing"] = (
            #     f"repoint{int(pointing.replace('p', '')):05d}"
            # )

            write_cdf(l1b_ds)
            # write_cdf(rates_ds)

            print(f"CDFs written for {pointing} pointing, sensor {id}\n")

[np.str_('p0'), np.str_('p1'), np.str_('p10'), np.str_('p100'), np.str_('p101'), np.str_('p102'), np.str_('p103'), np.str_('p104'), np.str_('p105'), np.str_('p106'), np.str_('p107'), np.str_('p108'), np.str_('p109'), np.str_('p11'), np.str_('p110'), np.str_('p111'), np.str_('p112'), np.str_('p113'), np.str_('p114'), np.str_('p115'), np.str_('p116'), np.str_('p117'), np.str_('p118'), np.str_('p119'), np.str_('p12'), np.str_('p120'), np.str_('p121'), np.str_('p122'), np.str_('p123'), np.str_('p124'), np.str_('p125'), np.str_('p126'), np.str_('p127'), np.str_('p128'), np.str_('p129'), np.str_('p13'), np.str_('p130'), np.str_('p131'), np.str_('p132'), np.str_('p133'), np.str_('p134'), np.str_('p135'), np.str_('p136'), np.str_('p137'), np.str_('p138'), np.str_('p139'), np.str_('p14'), np.str_('p140'), np.str_('p141'), np.str_('p142'), np.str_('p143'), np.str_('p144'), np.str_('p145'), np.str_('p146'), np.str_('p147'), np.str_('p148'), np.str_('p149'), np.str_('p15'), np.str_('p150'), np.str

ISTP Compliance Warning: Variable event_times does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable velocity_sc does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable velocity_dps_sc does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable velocity_dps_helio does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable energy_spacecraft does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable energy_heliosphere does not have an attribut

Datasets created
Coordinates:
  * epoch      (epoch) float64 138kB 7.98e+08 7.98e+08 ... 7.98e+08 7.98e+08
  * component  (component) <U1 12B 'x' 'y' 'z'
event_times
velocity_sc
velocity_dps_sc
velocity_dps_helio
energy_spacecraft
energy_heliosphere
energy
tof_energy
event_efficiency
geometric_factor_blades
quality_scattering
quality_outliers
scatter
ebin
theta
phi
epoch
component
CDFs written for p0 pointing, sensor 45

Creating direct event dataset for p1 pointing and 45 sensor
  Processing 17567 events
  Calculating theta/phi from instrument direction vectors (Java method)...
  Theta range: [-52.06, 51.90] deg
  Phi range: [-60.00, 59.94] deg

  === APPLYING SCATTERING FILTER ===
[(3.0, 3.4), (3.4, 3.8), (3.8, 4.2), (4.2, 4.6), (4.6, 5.19), (5.19, 5.78), (5.78, 6.37), (6.37, 6.96), (6.96, 7.7875), (7.7875, 8.615), (8.615, 9.4425), (9.4425, 10.27), (10.27, 11.63), (11.63, 12.99), (12.99, 14.35), (14.35, 15.71), (15.71, 17.3637), (17.3637, 19.1914), (19.1914, 21.2116), (21.2116, 23.44

ISTP Compliance Warning: Variable event_times does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable velocity_sc does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable velocity_dps_sc does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable velocity_dps_helio does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable energy_spacecraft does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable energy_heliosphere does not have an attribut

Datasets created
Coordinates:
  * epoch      (epoch) float64 141kB 7.981e+08 7.981e+08 ... 7.981e+08 7.981e+08
  * component  (component) <U1 12B 'x' 'y' 'z'
event_times
velocity_sc
velocity_dps_sc
velocity_dps_helio
energy_spacecraft
energy_heliosphere
energy
tof_energy
event_efficiency
geometric_factor_blades
quality_scattering
quality_outliers
scatter
ebin
theta
phi
epoch
component
CDFs written for p1 pointing, sensor 45

Creating direct event dataset for p10 pointing and 45 sensor
  Processing 16436 events
  Calculating theta/phi from instrument direction vectors (Java method)...
  Theta range: [-52.54, 52.09] deg
  Phi range: [-59.93, 59.98] deg

  === APPLYING SCATTERING FILTER ===
[(3.0, 3.4), (3.4, 3.8), (3.8, 4.2), (4.2, 4.6), (4.6, 5.19), (5.19, 5.78), (5.78, 6.37), (6.37, 6.96), (6.96, 7.7875), (7.7875, 8.615), (8.615, 9.4425), (9.4425, 10.27), (10.27, 11.63), (11.63, 12.99), (12.99, 14.35), (14.35, 15.71), (15.71, 17.3637), (17.3637, 19.1914), (19.1914, 21.2116), (21.2116, 

...

Datasets created
Coordinates:
  * epoch      (epoch) float64 92kB 8.064e+08 8.064e+08 ... 8.063e+08 8.064e+08
  * component  (component) <U1 12B 'x' 'y' 'z'
event_times
velocity_sc
velocity_dps_sc
velocity_dps_helio
energy_spacecraft
energy_heliosphere
energy
tof_energy
event_efficiency
geometric_factor_blades
quality_scattering
quality_outliers
scatter
ebin
theta
phi
epoch
component
CDFs written for p97 pointing, sensor 45

Creating direct event dataset for p98 pointing and 45 sensor
  Processing 11538 events
  Calculating theta/phi from instrument direction vectors (Java method)...
  Theta range: [-52.41, 51.48] deg
  Phi range: [-59.99, 59.97] deg

  === APPLYING SCATTERING FILTER ===
[(3.0, 3.4), (3.4, 3.8), (3.8, 4.2), (4.2, 4.6), (4.6, 5.19), (5.19, 5.78), (5.78, 6.37), (6.37, 6.96), (6.96, 7.7875), (7.7875, 8.615), (8.615, 9.4425), (9.4425, 10.27), (10.27, 11.63), (11.63, 12.99), (12.99, 14.35), (14.35, 15.71), (15.71, 17.3637), (17.3637, 19.1914), (19.1914, 21.2116), (21.2116, 

ISTP Compliance Warning: Variable event_times does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable velocity_sc does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable velocity_dps_sc does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable velocity_dps_helio does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable energy_spacecraft does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable energy_heliosphere does not have an attribut

Datasets created
Coordinates:
  * epoch      (epoch) float64 92kB 8.065e+08 8.065e+08 ... 8.064e+08 8.064e+08
  * component  (component) <U1 12B 'x' 'y' 'z'
event_times
velocity_sc
velocity_dps_sc
velocity_dps_helio
energy_spacecraft
energy_heliosphere
energy
tof_energy
event_efficiency
geometric_factor_blades
quality_scattering
quality_outliers
scatter
ebin
theta
phi
epoch
component
CDFs written for p98 pointing, sensor 45

Creating direct event dataset for p99 pointing and 45 sensor
  Processing 11340 events
  Calculating theta/phi from instrument direction vectors (Java method)...
  Theta range: [-51.51, 51.60] deg
  Phi range: [-59.98, 59.97] deg

  === APPLYING SCATTERING FILTER ===
[(3.0, 3.4), (3.4, 3.8), (3.8, 4.2), (4.2, 4.6), (4.6, 5.19), (5.19, 5.78), (5.78, 6.37), (6.37, 6.96), (6.96, 7.7875), (7.7875, 8.615), (8.615, 9.4425), (9.4425, 10.27), (10.27, 11.63), (11.63, 12.99), (12.99, 14.35), (14.35, 15.71), (15.71, 17.3637), (17.3637, 19.1914), (19.1914, 21.2116), (21.2116, 

ISTP Compliance Warning: Variable event_times does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable velocity_sc does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable velocity_dps_sc does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable velocity_dps_helio does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable energy_spacecraft does not have an attribute VAR_TYPE to describe the variable.  Attributes must be either data, support_data, metadata, or ignore_data.
ISTP Compliance Warning: Variable energy_heliosphere does not have an attribut

Datasets created
Coordinates:
  * epoch      (epoch) float64 91kB 8.065e+08 8.065e+08 ... 8.065e+08 8.065e+08
  * component  (component) <U1 12B 'x' 'y' 'z'
event_times
velocity_sc
velocity_dps_sc
velocity_dps_helio
energy_spacecraft
energy_heliosphere
energy
tof_energy
event_efficiency
geometric_factor_blades
quality_scattering
quality_outliers
scatter
ebin
theta
phi
epoch
component
CDFs written for p99 pointing, sensor 45

